In [ ]:
import os
import polars as pl
import pandas as pd
import pyranges as pr
from tqdm import tqdm
import matplotlib.pyplot as plt
from plotnine import *

In [ ]:
LOCAL_DATA_DIR = 'PATH_TO_FILE'
RAP_DATA_DIR = "project-REDACTED:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated"

# Add UKBBGym annotations

1. MAC and Sample count
2. VEP coding / non-coding
4. Amino acid position
5. Protein domains (MOBI db)

In [ ]:
# Download variant metadata file

!dx download {RAP_DATA_DIR}/qced_maf1e-3_loftee_olink_genes_EURunrelated_variant_metadata.parquet -o {LOCAL_DATA_DIR}/

varmeta_df = pl.read_parquet(f'{LOCAL_DATA_DIR}/qced_maf1e-3_loftee_olink_genes_EURunrelated_variant_metadata.parquet')
varmeta_df.head()

In [ ]:
# Download annotations file
!dx download {RAP_DATA_DIR}/annotations.parquet -o {LOCAL_DATA_DIR}/

anno_df = pl.scan_parquet(f'{LOCAL_DATA_DIR}/annotations.parquet')
anno_df.head().collect()

In [ ]:
# Download annotations fill_na file
!dx download {RAP_DATA_DIR}/annotations_fill_na.parquet -o {LOCAL_DATA_DIR}/

anno_fillna_df = pl.scan_parquet(f'{LOCAL_DATA_DIR}/annotations_fill_na.parquet')
anno_fillna_df.head().collect()

---
# Get AbExp min, and max; drop the individuals abexp tissue columns

In [ ]:
ab_exp_tissue_cols = [c for c in anno_fillna_df.columns if (c.startswith('abexp_') and not c.endswith('_is_na'))]
ab_exp_tissue_cols = list(set(ab_exp_tissue_cols) - {'abexp_abs_max'})
ab_exp_tissue_cols

In [ ]:
abexp_df = (
    anno_fillna_df
    .with_columns(
        abexp_min = pl.min_horizontal([pl.col(c) for c in ab_exp_tissue_cols]),
        abexp_max = pl.max_horizontal([pl.col(c) for c in ab_exp_tissue_cols]),
        abexp_mean = pl.mean_horizontal([pl.col(c) for c in ab_exp_tissue_cols]),
    )
    .select(['id', 'region', 'abexp_abs_max', 'abexp_min', 'abexp_max', 'abexp_mean'])
    .collect()
)

abexp_df

In [ ]:
ab_exp_cols = [c for c in anno_fillna_df.columns if c.startswith('abexp_')]

drop_abexp_cols = list(set(ab_exp_cols) - {'abexp_abs_max', 'abexp_abs_max_is_na'})
drop_abexp_cols += ['ac_ukb_eur', 'mac_ukb_eur', 'mac_ukb']

anno_fillna_df = (
    anno_fillna_df
    .drop(drop_abexp_cols + ['spliceai_pred'])
    .join(abexp_df.lazy(), on=['id', 'region'], how='left')
)

anno_fillna_df.columns

---
# Annotate UKB MAC and sample counts

In [ ]:
anno_fillna_df = (
    anno_fillna_df

    .join(
        varmeta_df.select(['id', 'sc_ukb', 'ac_ukb', 'mac_ukb']).lazy(), 
        on='id', 
        how='inner'
    )
)

---
# Annotate VEP coding regions

In [ ]:
vep_cds_relaxed = [
    # LoF variants
    'consequence_stop_gained',
    'consequence_frameshift_variant',

    # Missense and synonymous variants
    'consequence_synonymous_variant',
    'consequence_protein_altering_variant',
    'consequence_missense_variant',

    # Splicing variants
    'consequence_splice_region_variant',
    'consequence_splice_acceptor_variant',
    'consequence_splice_donor_variant',
    'consequence_splice_donor_region_variant',
    'consequence_splice_donor_5th_base_variant',
    'consequence_splice_polypyrimidine_tract_variant',

    # Other coding sequence variants
    'consequence_start_lost',
    'consequence_stop_lost',
    'consequence_start_retained_variant',
    'consequence_stop_retained_variant',
    'consequence_inframe_deletion',
    'consequence_inframe_insertion',
    'consequence_coding_sequence_variant',
]

anno_fillna_df = (
    anno_fillna_df
    .with_columns(
        vep_cds_relaxed = pl.sum_horizontal(vep_cds_relaxed) > 0,
    )
)

---
# MANE CDS & non-MANE CDS only

## 1. Parse GTF & extract MANE / non-MANE CDS regions

In [ ]:
gtf_path = "/s/genomes/Gencode/Gencode_human/release_40/gencode.v40.annotation.gtf.gz"

gtf_pl = pl.read_csv(
    gtf_path,
    separator="\t",
    comment_prefix="#",
    has_header=False,
    new_columns=["Chromosome", "source", "Feature", "Start", "End", "score", "Strand", "frame", "attributes"]
).with_row_index("row_nr")

# Parse attributes into columns
atts = (
    gtf_pl.select("row_nr", "attributes")
    .with_columns(attrs_list=pl.col("attributes").str.split("; "))
    .explode("attrs_list")
    .with_columns(
        pl.col("attrs_list").str.split_exact(" ", 1)
        .struct.rename_fields(["attribute", "value"]).alias("fields")
    ).unnest("fields")
    .with_columns(pl.col("value").str.strip_chars('"'))
    .pivot(index="row_nr", on="attribute", values="value", aggregate_function=pl.element().implode())
    .with_columns(pl.col(pl.List).list.join(", ").str.strip_chars(";").replace("", None))
    .with_columns(level=pl.col('level').cast(pl.Int32))
)

gtf = gtf_pl.join(atts, on='row_nr')
gtf

In [ ]:
# Filter to protein-coding CDS features only
cds = gtf.filter(
    (pl.col('gene_type') == 'protein_coding') & (pl.col('Feature') == 'CDS')
)

# MANE CDS: merged regions from MANE_Select transcripts
mane_cds_pr = pr.PyRanges(
    cds.filter(pl.col('tag').str.contains('MANE_Select')).to_pandas()
).merge()

# Non-MANE CDS: merged regions from non-MANE transcripts, subtracting MANE CDS
non_mane_cds_pr = pr.PyRanges(
    cds.filter(~pl.col('tag').str.contains('MANE_Select')).to_pandas()
).merge().subtract(mane_cds_pr)

print(f"MANE CDS regions: {len(mane_cds_pr)}")
print(f"Non-MANE CDS regions: {len(non_mane_cds_pr)}")

## 2. Load annotations & overlap with CDS regions

In [ ]:
# anno_df = pl.scan_parquet(f'{LOCAL_DATA_DIR}/annotations_fillna_ukbgym.parquet')
anno_fillna_df.head().collect()

In [ ]:
# Convert annotations to PyRanges for overlap
anno = (
    anno_fillna_df
    .with_columns(
        Chromosome=pl.col('chrom'),
        Start=pl.col('pos') - 1,
        End=pl.col('pos') - 1 + pl.col('ref').str.len_chars(),
        gene=pl.col('region'),
    )
    .select(['id', 'Chromosome', 'Start', 'End', 'ref', 'alt', 'gene'])
    .collect(engine='streaming')
)
anno_pr = pr.PyRanges(anno.to_pandas())

# Overlap with MANE and non-MANE CDS regions
mane_cds_vars = pl.from_pandas(anno_pr.overlap(mane_cds_pr).df).with_columns(mane_cds=pl.lit(True)).select(['id', 'gene', 'mane_cds'])
non_mane_cds_vars = pl.from_pandas(anno_pr.overlap(non_mane_cds_pr).df).with_columns(non_mane_cds=pl.lit(True)).select(['id', 'gene', 'non_mane_cds'])

mane_cds_vars

## 3. Join flags back to annotations & save

In [ ]:
# Join MANE/non-MANE CDS flags to the annotation dataframe
mane_annos = (
    anno_fillna_df
    .with_columns(gene=pl.col('region').cast(pl.Utf8))
    .select(['id', 'gene'])
    .collect(engine='streaming')
    .join(mane_cds_vars, on=['id', 'gene'], how='left')
    .join(non_mane_cds_vars, on=['id', 'gene'], how='left')
    .fill_null(False)
    .unique()
    .rename({'gene': 'region'})
)

anno_fillna_df = anno_fillna_df.join(mane_annos.lazy(), on=['id', 'region'], how='left').fill_null(False)
anno_fillna_df.sink_parquet(f'{LOCAL_DATA_DIR}/annotations_with_mane.parquet')
anno_fillna_df.head().collect()

---
# Annotate protein domains

## Merge protein position, amino acid mutation

In [ ]:
anno_fillna_df = pl.scan_parquet(f'{LOCAL_DATA_DIR}/annotations_with_mane.parquet')
anno_fillna_df.head().collect()

In [ ]:
anno_ukbgym_df = (
    anno_fillna_df
    
    .join(
        anno_df.select(['id', 'region', 'protein_position', 'amino_acids']).lazy(), 
        on='id', 
        how='inner'
    )
)

## Annotate protein domains

In [ ]:
anno_coding = (
    anno_ukbgym_df
    .select(['id', 'region', 'protein_position'])
    .drop_nulls()
    .with_columns(
        pos_raw = pl.col('protein_position').str.split('/').list.get(0)
    )
    .with_columns(
        split_struct = pl.col('pos_raw').str.split_exact('-', 1)
    )
    .with_columns(
        aa_start = pl.col('split_struct').struct.field('field_0').cast(pl.Int32, strict=False),
        aa_end = (
            pl.col('split_struct').struct.field('field_1')
            .fill_null(pl.col('split_struct').struct.field('field_0'))
            .cast(pl.Int32, strict=False)
        )
    )
    .drop(['pos_raw', 'split_struct'])
    .collect(engine='streaming')
)

anno_coding

In [ ]:
ginfo = (
    pl.read_parquet('PATH_TO_FILE')
    .filter(pl.col('ensembl_canonical') == True)
    .select(['gene_stable_id', 'gene_name', 'uniprotkb_gene_name_id', 'uniprotkb_gene_name_symbol', 'gene_type'])
    .rename({
        'gene_stable_id': 'region',
        'uniprotkb_gene_name_id': 'uniprot_id',
        'uniprotkb_gene_name_symbol': 'uniprot_name'
    })
    .filter(pl.col('gene_type') == 'protein_coding')
)
ginfo

## Disorder - MobiDB

In [ ]:
mobi_df_raw = pl.read_csv(
    "PATH_TO_FILE",
    separator="\t",
    has_header=False,
    new_columns=[
        "uniprot_id",
        "feature",
        "protein_regions",
        "disorder_content",
        "disorder_count",
        "length"
    ]
)

mobi_df_raw

In [ ]:
a = mobi_df_raw.filter(pl.col('uniprot_id')=='P04637').filter(pl.col('feature').str.contains('disorder'))
a

In [ ]:
mobi_df = (
    mobi_df_raw
    .join(ginfo.select(['uniprot_id', 'region']).unique(), on='uniprot_id', how='inner')  # Keep only proteins present in ginfo
    # .join(anno_coding.select(['region']).unique(), on='region', how='semi')  # Keep only proteins present in anno_coding

    .with_columns(pl.col("protein_regions").str.split(",")) # Split "1..2,71..75" into list
    .explode("protein_regions")                             # Create new row for each region
    .with_columns(
        # Split "1..2" into separate Start and End columns
        pl.col("protein_regions")
        .str.split_exact("..", 1)
        .struct.rename_fields(["domain_start", "domain_end"])
        .alias("protein_region_struct")
    )
    .unnest("protein_region_struct")
    .with_columns([
        pl.col("domain_start").cast(pl.Int64),
        pl.col("domain_end").cast(pl.Int64)
    ])

    .with_columns(
        mobi_feature_source = pl.col("feature").str.split("-").list.get(0),
        mobi_feature_type = pl.col("feature").str.split("-").list.get(1),
        mobi_feature_subtype = pl.col("feature").str.split("-").list.get(2)
    )

    .with_columns(
        mobi_full_disorder_priority = (pl.col('mobi_feature_source')=='curated') & (pl.col('mobi_feature_type')=='disorder'),
        mobi_curated_disorder_priority = (pl.col('mobi_feature_source')=='curated') & (pl.col('mobi_feature_type')=='disorder') & (pl.col('mobi_feature_subtype')=='priority'),
        # mobi_lip_full = (pl.col('mobi_feature_type')=='lip') & (pl.col('mobi_feature_subtype')=='priority'),
        mobi_full_lip_priority = (pl.col('mobi_feature_type')=='lip') & (pl.col('mobi_feature_subtype')=='priority'),
    )

    .filter(
        (pl.col('mobi_full_disorder_priority') == True) |
        (pl.col('mobi_curated_disorder_priority') == True) |
        (pl.col('mobi_full_lip_priority') == True)
    )

    .select(['region', 'uniprot_id', 'domain_start', 'domain_end', 'mobi_full_disorder_priority', 'mobi_curated_disorder_priority', 'mobi_full_lip_priority'])
    .unique()
    .sort(['uniprot_id', 'domain_start'])
)

mobi_df

## Structured domains - TED

In [ ]:
ted_df_raw = pl.read_csv('PATH_TO_FILE', separator='\t', has_header=False)
ted_df_raw

In [ ]:
ted_df = (
    ted_df_raw
    .with_columns(
        uniprot_id = pl.col('column_1').str.split('-').list.get(1),
        ted_id = pl.col('column_1').str.slice(-5),
    )
    .with_columns(pl.col("column_4").str.split("_")) # Split "1-2_71-75" into list
    .explode("column_4")                             # Create new row for each region
    .with_columns(
        # Split "1-2" into separate Start and End columns
        pl.col("column_4")
        .str.split_exact("-", 1)
        .struct.rename_fields(["domain_start", "domain_end"])
        .alias("protein_region_struct")
    )
    .unnest("protein_region_struct")
    .with_columns(
        [
            pl.col("domain_start").cast(pl.Int64),
            pl.col("domain_end").cast(pl.Int64)
        ],
        ted_domain = pl.lit(True)
    )
    .select(['uniprot_id', 'domain_start', 'domain_end', 'ted_domain'])
    .unique()
    .join(ginfo.select(['uniprot_id', 'region']).unique(), on='uniprot_id', how='inner')  # Keep only proteins present in ginfo
)

ted_df

## Low complexity regions

In [ ]:
lc_df_raw = pl.read_csv('PATH_TO_FILE', separator='\t')
lc_df_raw

In [ ]:
lc_df = (
    lc_df_raw
    .filter(pl.col('Organism')=='H. sapiens')
    .rename({
        'Protein ID': 'uniprot_id',
    })
    .with_columns(
        # Split "1-2" into separate Start and End columns
        pl.col("Domain Boundaries")
        .str.strip_chars("()")
        .str.split_exact("-", 1)
        .struct.rename_fields(["domain_start", "domain_end"])
        .alias("protein_region_struct")
    )
    .unnest("protein_region_struct")
    .with_columns(
        [
            pl.col("domain_start").cast(pl.Int64),
            pl.col("domain_end").cast(pl.Int64)
        ],
        low_complexity_domain = pl.lit(True)
    )
    .select(['uniprot_id', 'domain_start', 'domain_end', 'low_complexity_domain'])
    .unique()
    .join(ginfo.select(['uniprot_id', 'region']).unique(), on='uniprot_id', how='inner')  # Keep only proteins present in ginfo
)

lc_df

## Merge with annotations

In [ ]:
all_df = pl.concat([mobi_df, ted_df, lc_df], how='diagonal').fill_null(False)
all_df

In [ ]:
overlap_df = (
    anno_coding.lazy() # Use lazy for better performance on large joins
    .join(
        all_df.lazy(), 
        on="region", 
        how="inner"
    )
    .filter(
        # (Variant Start <= Region End) AND (Variant Start >= Region Start)
        ((pl.col("aa_start") <= pl.col("domain_end")) & (pl.col("aa_start") >= pl.col("domain_start"))) |

        # (Variant End <= Region End) AND (Variant End >= Region Start)
        ((pl.col("aa_end") <= pl.col("domain_end")) & (pl.col("aa_end") >= pl.col("domain_start")))
    )
    .group_by(['id', 'region', 'aa_start', 'aa_end'])
    .agg([
        pl.col('mobi_full_disorder_priority').max().alias('mobi_full_disorder_priority'),
        pl.col('mobi_curated_disorder_priority').max().alias('mobi_curated_disorder_priority'),
        pl.col('mobi_full_lip_priority').max().alias('mobi_full_lip_priority'),
        pl.col('ted_domain').max().alias('ted_domain'),
        pl.col('low_complexity_domain').max().alias('low_complexity_domain'),
    ])
    .collect()
)

overlap_df

In [ ]:
all_df = (
    anno_ukbgym_df.join(
        overlap_df.drop(['aa_start', 'aa_end']).lazy(),
        on=['id', 'region'],
        how='left'
    )
    .with_columns(
        pl.col('mobi_full_disorder_priority').fill_null(False),
        pl.col('mobi_curated_disorder_priority').fill_null(False),
        pl.col('mobi_full_lip_priority').fill_null(False),
        pl.col('ted_domain').fill_null(False),
        pl.col('low_complexity_domain').fill_null(False)
    )
    .unique()
    # .collect(engine='streaming')
)

all_df.sink_parquet(f'{LOCAL_DATA_DIR}/annotations_fillna_ukbgym.parquet', engine='streaming')

---
# Add REVEL

https://sites.google.com/site/revelgenomics/

In [ ]:
rev = (
    pl.scan_csv(
        'PATH_TO_FILE', 
        null_values=['.'],
        schema_overrides={
            'chr': pl.Utf8,
        }
    )
    .with_columns(
        chrom = 'chr' + pl.col('chr'),
    )
    .rename({
        'grch38_pos': 'pos',
        'REVEL': 'revel_score'
    })
    .select(
        ['chrom', 'pos', 'ref', 'alt', 'revel_score', 'Ensembl_transcriptid']
    )
)

rev.head().collect()

In [ ]:
# Build transcript_id -> gene_id map from GTF
gtf_path = "/s/genomes/Gencode/Gencode_human/release_40/gencode.v40.annotation.gtf.gz"

transcript_gene_map = (
    pl.read_csv(
        gtf_path, separator="\t", comment_prefix="#", has_header=False,
        new_columns=["Chromosome", "source", "Feature", "Start", "End", "score", "Strand", "frame", "attributes"]
    )
    .filter(pl.col('Feature') == 'transcript')
    .select(
        gene_id=pl.col('attributes').str.extract(r'gene_id "([^"]+)"').str.split('.').list.get(0),
        transcript_id=pl.col('attributes').str.extract(r'transcript_id "([^"]+)"').str.split('.').list.get(0),
    )
    .unique()
)

print(f"Unique transcript->gene mappings: {len(transcript_gene_map)}")
transcript_gene_map.head()

In [ ]:
# Join gene IDs to REVEL via transcript_id -> gene_id map
# Ensembl_transcriptid is semicolon-separated (e.g. "ENST00000359596;ENST00000355489")
# -> split and explode before joining to the transcript->gene map
rev_with_genes = (
    rev
    .with_columns(
        pl.col('Ensembl_transcriptid').str.split(';')
    )
    .explode('Ensembl_transcriptid')
    .join(
        transcript_gene_map.lazy().rename({'transcript_id': 'Ensembl_transcriptid'}),
        on='Ensembl_transcriptid',
        how='inner'
    )
    .rename({'gene_id': 'region'})
    .select(['region', 'chrom', 'pos', 'ref', 'alt', 'revel_score'])
    .group_by(['region', 'chrom', 'pos', 'ref', 'alt'])
    .agg(pl.col('revel_score').max())  # take max score across transcripts per gene
)

rev_with_genes.head().collect()

In [ ]:
# Join REVEL scores to annotations on region, chrom, pos, ref, alt
anno_with_revel = (
    pl.scan_parquet(f'{LOCAL_DATA_DIR}/annotations_fillna_ukbgym.parquet')
    .join(
        rev_with_genes,
        on=['region', 'chrom', 'pos', 'ref', 'alt'],
        how='left'
    )
)

anno_with_revel.head().collect()

In [ ]:
# Check coverage
total = anno_with_revel.select(pl.len()).collect().item()
with_revel = anno_with_revel.filter(pl.col('revel_score').is_not_null()).select(pl.len()).collect().item()
print(f"Variants with REVEL scores: {with_revel} / {total} ({100*with_revel/total:.1f}%)")

In [ ]:
anno_with_revel.filter(pl.col('consequence_missense_variant')==1).collect()

---
# Add ESM1b

https://huggingface.co/spaces/ntranoslab/esm_variants/tree/main

In [ ]:
anno_with_revel.head().collect()

In [ ]:
# Get uniprot_ids for genes present in the annotations
region_to_uniprot = (
    ginfo
    .select(['gene_stable_id', 'uniprotkb_gene_name_id'])
    .rename({
        'gene_stable_id': 'region',
        'uniprotkb_gene_name_id': 'uniprot_id'
    })
    .unique()
    .join(
        anno_with_revel.select('region').unique().collect(),
        on='region',
        how='semi'  # keep only regions that exist in anno
    )
)

needed_uniprots = set(region_to_uniprot['uniprot_id'].to_list())

# Check which have CSV files
esm_csv_dir = f"{ESM1B_DIR}/content/ALL_hum_isoforms_ESM1b_LLR"
esm_files_to_read = {
    uid: f"{esm_csv_dir}/{uid}_LLR.csv"
    for uid in needed_uniprots
    if os.path.exists(f"{esm_csv_dir}/{uid}_LLR.csv")
}

print(f"Proteins in anno: {len(needed_uniprots)}")
print(f"ESM1b CSV files available: {len(esm_files_to_read)}")

In [ ]:
# Read raw CSVs for needed proteins, unpivot to long format, and map uniprot_id -> region
esm_dfs = []
for uid, path in tqdm(esm_files_to_read.items()):
    df = (
        pl.read_csv(path)
        .rename({'': 'alt_aa'})
        .unpivot(index='alt_aa', value_name='esm1b_llr', variable_name='position')
        .with_columns(
            ref_aa=pl.col('position').str.split_exact(' ', 1).struct.field('field_0'),
            prot_pos=pl.col('position').str.split_exact(' ', 1).struct.field('field_1'),
        )
        .with_columns(
            amino_acids=pl.col('ref_aa') + '/' + pl.col('alt_aa'),
            uniprot_id=pl.lit(uid),
        )
        .select(['uniprot_id', 'amino_acids', 'prot_pos', 'esm1b_llr'])
    )
    esm_dfs.append(df)

esm_scores = (
    pl.concat(esm_dfs)
    .join(region_to_uniprot, on='uniprot_id', how='inner')
    .select(['region', 'amino_acids', 'prot_pos', 'esm1b_llr'])
)

print(f"Total ESM1b scores: {len(esm_scores):,}")
esm_scores.head()

In [ ]:
# Join ESM1b scores to annotations
# protein_position in anno is "pos/protein_length" (e.g. "741/747")
# prot_pos in ESM1b is just the position (e.g. "741")
# -> extract position part from anno before joining
anno_with_esm = (
    anno_with_revel
    .with_columns(
        prot_pos=pl.col('protein_position').str.split('/').list.get(0)
    )
    .join(
        esm_scores.lazy(),
        on=['region', 'amino_acids', 'prot_pos'],
        how='left'
    )
    .drop('prot_pos')
)

anno_with_esm.head().collect()

In [ ]:
# Check coverage
total = anno_with_esm.select(pl.len()).collect().item()
with_esm = anno_with_esm.filter(pl.col('esm1b_llr').is_not_null()).select(pl.len()).collect().item()
print(f"Variants with ESM1b scores: {with_esm} / {total} ({100*with_esm/total:.1f}%)")

---
# Add CPT-1

https://huggingface.co/spaces/songlab/CPT/tree/main

In [ ]:
CPT1_DIR = "PATH_TO_FILE"

# Build entry_name -> uniprot_accession mapping
uniprot_entry_map = (
    pl.read_csv(f"{ESM1B_DIR}/content/ALL_hum_isoforms_ESM1b_LLR/000_uniprot_df.csv")
    .select(
        uniprot_id=pl.col('id'),        # accession e.g. Q16348
        entry_name=pl.col('gene'),       # entry name e.g. S15A2
    )
)

# Chain: entry_name -> uniprot_accession -> region (gene_id)
cpt1_entry_to_region = (
    uniprot_entry_map
    .join(region_to_uniprot, on='uniprot_id', how='inner')  # adds 'region'
    .select(['entry_name', 'region'])
    .unique()
)

print(f"Entry names mapped to anno regions: {len(cpt1_entry_to_region)}")
cpt1_entry_to_region.head()

In [ ]:
# Read CPT1 scores only for needed proteins, parse mutant column
needed_entries = set(cpt1_entry_to_region['entry_name'].to_list())

cpt1_dfs = []
for entry_name in tqdm(needed_entries):
    path = f"{CPT1_DIR}/CPT1_all_proteins/{entry_name}_HUMAN.csv.gz"
    if not os.path.exists(path):
        continue
    df = (
        pl.read_csv(path)
        .rename({'CPT1_score': 'cpt1_llr'})
        .with_columns(
            # Parse "M1A" -> ref_aa="M", pos="1", alt_aa="A"
            ref_aa=pl.col('mutant').str.slice(0, 1),
            alt_aa=pl.col('mutant').str.slice(-1, 1),
            prot_pos=pl.col('mutant').str.slice(1, pl.col('mutant').str.len_chars() - 2),
        )
        .with_columns(
            amino_acids=pl.col('ref_aa') + '/' + pl.col('alt_aa'),
            entry_name=pl.lit(entry_name),
        )
        .select(['entry_name', 'amino_acids', 'prot_pos', 'cpt1_llr'])
    )
    cpt1_dfs.append(df)

cpt1_scores = (
    pl.concat(cpt1_dfs)
    .join(cpt1_entry_to_region, on='entry_name', how='inner')
    .select(['region', 'amino_acids', 'prot_pos', 'cpt1_llr'])
)

print(f"Total CPT1 scores: {len(cpt1_scores):,}")
cpt1_scores.head()

In [ ]:
# Join CPT1 scores to annotations
# Same approach: extract position part from protein_position before joining
anno_with_cpt1 = (
    anno_with_esm
    .with_columns(
        prot_pos=pl.col('protein_position').str.split('/').list.get(0)
    )
    .join(
        cpt1_scores.lazy(),
        on=['region', 'amino_acids', 'prot_pos'],
        how='left'
    )
    .drop('prot_pos')
)

anno_with_cpt1.head().collect()

In [ ]:
# Check coverage
total = anno_with_cpt1.select(pl.len()).collect().item()
with_cpt1 = anno_with_cpt1.filter(pl.col('cpt1_llr').is_not_null()).select(pl.len()).collect().item()
print(f"Variants with CPT1 scores: {with_cpt1} / {total} ({100*with_cpt1/total:.1f}%)")

In [ ]:
anno_with_cpt1.filter(pl.col('consequence_missense_variant')==1).collect()

---
# Add ClinPred

https://sites.google.com/site/clinpred/home

In [ ]:
# TEMPORARY, USE PARTIALLY ANNOTATED DF

RAP_ANNO_DIR = "project-REDACTED:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated"

ANNO_FILE = "annotations_with_all.parquet"

!dx download {RAP_ANNO_DIR}/{ANNO_FILE} -o {LOCAL_DATA_DIR}/{ANNO_FILE}

anno = (
    pl.scan_parquet(f"{LOCAL_DATA_DIR}/{ANNO_FILE}")
    .with_columns(
        cpt1_llr_is_na = pl.when(pl.col('cpt1_llr').is_null()).then(1).otherwise(0),
        esm1b_llr_is_na = pl.when(pl.col('esm1b_llr').is_null()).then(1).otherwise(0),
        revel_score_is_na = pl.when(pl.col('revel_score').is_null()).then(1).otherwise(0)
    )
    .with_columns(
        cpt1_llr = pl.col('cpt1_llr').fill_null(0.0).cast(pl.Float32),  # fill missing CPT1 scores with 0 (neutral)
        esm1b_llr = pl.col('esm1b_llr').fill_null(0.0).cast(pl.Float32),  # fill missing ESM1b scores with 0 (neutral)	
        revel_score =  pl.col('revel_score').fill_null(0.0).cast(pl.Float32)  # fill missing REVEL scores with 0 (neutral)
    )
)

anno.head().collect()

In [ ]:
cp = (
    pl.scan_csv(
        "PATH_TO_FILE", 
        separator='\t',
        schema_overrides={
            'Chr': pl.Utf8,
            'ClinPred_Score': pl.Float32
        },
        ignore_errors=True
    )
    .with_columns(
        chrom = 'chr' + pl.col('Chr').cast(pl.Utf8),
    )
    .rename({
        'Start': 'pos',
        'Ref': 'ref',
        'Alt': 'alt',
        'ClinPred_Score': 'clinpred_score'
    })
    .drop('Chr')
)

cp.head().collect()

In [ ]:
plt.hist(cp.head(10_000).select('clinpred_score').collect(), bins=50)
plt.show()

In [ ]:
anno_w_cp = (
    anno
    .join(
        cp.lazy(),
        on=['chrom', 'pos', 'ref', 'alt'],
        how='left'
    )
    .with_columns(
        clinpred_score_is_na = pl.when(pl.col('clinpred_score').is_null()).then(1).otherwise(0),
    )
)

anno_w_cp.head().collect()

In [ ]:
# Check coverage
total = anno_w_cp.select(pl.len()).collect().item()
with_cp = anno_w_cp.filter(pl.col('clinpred_score').is_not_null()).select(pl.len()).collect().item()
print(f"Variants with ClinPred scores: {with_cp} / {total} ({100*with_cp/total:.1f}%)")

anno_w_cp.filter(pl.col('consequence_missense_variant')==1).collect()

In [ ]:
anno_w_cp = (
    anno_w_cp
    .with_columns(
        clinpred_score = pl.col('clinpred_score').fill_null(0.0).cast(pl.Float32),
    )
    .collect()
)

In [ ]:
anno_w_cp.write_parquet(f"{LOCAL_DATA_DIR}/annotations_with_all_with_clinpred.parquet")

---
# Add BayesDel

https://fenglab.chpc.utah.edu/BayesDel/BayesDel.html

In [ ]:
bd = (
    pl.scan_csv(
        "PATH_TO_FILE", 
        separator='\t',
        schema_overrides={
            '#Chr': pl.Utf8,
        },
        ignore_errors=True
    )
    .with_columns(
        chrom = 'chr' + pl.col('#Chr').cast(pl.Utf8),
    )
    .rename({
        'Start': 'pos',
        'BayesDel_nsfp33a_noAF': 'bayes_del'
    })
    .drop('#Chr')
)

bd.head().collect()

In [ ]:
plt.hist(bd.head(10_000).select('bayes_del').collect(), bins=50)
plt.show()

In [ ]:
anno_w_cp = pl.scan_parquet(f"{LOCAL_DATA_DIR}/annotations_with_all_with_clinpred.parquet")

anno_w_bd = (
    anno_w_cp
    .join(
        bd.lazy(),
        on=['chrom', 'pos', 'ref', 'alt'],
        how='left'
    )
    .with_columns(
        bayes_del_is_na = pl.when(pl.col('bayes_del').is_null()).then(1).otherwise(0),
    )
)

anno_w_bd.head().collect()

In [ ]:
# Check coverage
total = anno_w_bd.select(pl.len()).collect().item()
with_bd = anno_w_bd.filter(pl.col('bayes_del').is_not_null()).select(pl.len()).collect().item()
print(f"Variants with BayesDel scores: {with_bd} / {total} ({100*with_bd/total:.1f}%)")

anno_w_bd.filter(pl.col('consequence_missense_variant')==1).collect()

In [ ]:
bd_median = bd.select('bayes_del').collect().median().item()
bd_median

In [ ]:
anno_w_bd = (
    anno_w_bd
    .with_columns(
        bayes_del = pl.col('bayes_del').fill_null(bd_median).cast(pl.Float32),
    )
    .collect()
)

In [ ]:
anno_w_bd.write_parquet(f"{LOCAL_DATA_DIR}/annotations_with_all_with_clinpred_bayesdel.parquet")

---
# Add popEVE

https://pop.evemodel.org/documentation

`wget https://data.evemodel.org/popeve/v1.1/downloads/grch38_popEVE_ukbb_20250715.vcf.gz`

In [ ]:
pe = (
    pl.read_csv(
        'PATH_TO_FILE', 
        separator='\t',
        ignore_errors=True,
    )
    .rename({
        '#CHROM': 'chrom',
        'POS': 'pos',
        'REF': 'ref',
        'ALT': 'alt',
    })
    .select(['chrom', 'pos', 'ref', 'alt', 'INFO'])
    .with_columns(
        # Extract everything after 'protein=' until the next ';' or end of string
        protein = pl.col('INFO').str.extract(r"protein=([^;]+)"),
        
        # Extract everything after 'gene=' until the next ';' or end of string
        gene_name = pl.col('INFO').str.extract(r"gene=([^;]+)"),
        mutant = pl.col('INFO').str.extract(r"mutant=([^;]+)"),
        
        popeve_score = pl.col('INFO').str.extract(r"popEVE=([^;]+)").cast(pl.Float32),
        eve_score = pl.col('INFO').str.extract(r"EVE=([^;]+)").cast(pl.Float32),
        pop_adj_eve = pl.col('INFO').str.extract(r"pop-adjusted_EVE=([^;]+)").cast(pl.Float32),
        pop_adj_esm1v = pl.col('INFO').str.extract(r"pop-adjusted_ESM1v=([^;]+)").cast(pl.Float32),
    )
    .with_columns(
        ref_aa=pl.col('mutant').str.slice(0, 1),
        alt_aa=pl.col('mutant').str.slice(-1, 1),
        prot_pos=pl.col('mutant').str.slice(1, pl.col('mutant').str.len_chars() - 2),
    )
    .with_columns(
        amino_acids=pl.col('ref_aa') + '/' + pl.col('alt_aa'),
    )
    .select(['gene_name', 'amino_acids', 'prot_pos', 'popeve_score', 'eve_score', 'pop_adj_eve', 'pop_adj_esm1v'])
)

pe

In [ ]:
plt.hist(pe['popeve_score'].head(10_000), bins=50)
plt.show()

In [ ]:
ginfo = (
    pl.read_parquet('PATH_TO_FILE')
    .filter(pl.col('gene_type') == 'protein_coding')
    .select(['gene_stable_id', 'gene_name'])
    .rename({'gene_stable_id': 'region'})
    .unique()
)
ginfo

In [ ]:
len(set(pe.select(['gene_name']).unique()['gene_name']) - set(ginfo.select(['gene_name']).unique()['gene_name']))

In [ ]:
anno_w_bd = (
    pl.scan_parquet(f"{LOCAL_DATA_DIR}/annotations_with_all_with_clinpred_bayesdel.parquet")
    .join(
        ginfo.lazy(),
        on='region',
        how='inner'
    )
)
anno_w_bd.head().collect()

In [ ]:
anno_with_popeve = (
    anno_w_bd
    .with_columns(
        prot_pos=pl.col('protein_position').str.split('/').list.get(0)
    )
    .join(
        pe.lazy(),
        on=['gene_name', 'amino_acids', 'prot_pos'],
        how='left'
    )
    .with_columns(
        popeve_score_is_na = pl.when(pl.col('popeve_score').is_null()).then(0).otherwise(1),
        eve_score_is_na = pl.when(pl.col('eve_score').is_null()).then(0).otherwise(1),
        pop_adj_eve_is_na = pl.when(pl.col('pop_adj_eve').is_null()).then(0).otherwise(1),
        pop_adj_esm1v_is_na = pl.when(pl.col('pop_adj_esm1v').is_null()).then(0).otherwise(1),
    )
    .drop('prot_pos')
    .collect()
)

anno_with_popeve

In [ ]:
# Check coverage
total = anno_with_popeve.select(pl.len()).item()
with_pe = anno_with_popeve.filter(pl.col('popeve_score').is_not_null()).select(pl.len()).item()
print(f"Variants with popEVE scores: {with_pe} / {total} ({100*with_pe/total:.1f}%)")

In [ ]:
popeve_score_median = pe.select('popeve_score').median().item()
eve_score_median = pe.select('eve_score').median().item()
pop_adj_eve_median = pe.select('pop_adj_eve').median().item()
pop_adj_esm1v_median = pe.select('pop_adj_esm1v').median().item()

print(f"popEVE score median: {popeve_score_median}")
print(f"EVE score median: {eve_score_median}")
print(f"pop-adjusted EVE score median: {pop_adj_eve_median}")
print(f"pop-adjusted ESM1v score median: {pop_adj_esm1v_median}")

In [ ]:
(
    anno_with_popeve
    .with_columns(
        popeve_score = pl.col('popeve_score').fill_null(popeve_score_median).cast(pl.Float32),
        eve_score = pl.col('eve_score').fill_null(eve_score_median).cast(pl.Float32),
        pop_adj_eve = pl.col('pop_adj_eve').fill_null(pop_adj_eve_median).cast(pl.Float32),
        pop_adj_esm1v = pl.col('pop_adj_esm1v').fill_null(pop_adj_esm1v_median).cast(pl.Float32),
    )
    .write_parquet(f"{LOCAL_DATA_DIR}/annotations_with_all_with_clinpred_bayesdel_popeve.parquet")
)

---
# Add ClinVar

In [ ]:
LOCAL_ANNO_DIR = "PATH_TO_FILE"

# Download ClinVar VCF (GRCh38) and its index
!mkdir -p {LOCAL_ANNO_DIR}/clinvar
!wget -nc https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz -O {LOCAL_ANNO_DIR}/clinvar/clinvar.vcf.gz
!wget -nc https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz.tbi -O {LOCAL_ANNO_DIR}/clinvar/clinvar.vcf.gz.tbi

In [ ]:
clinvar = (
    pl.read_csv(f"{LOCAL_ANNO_DIR}/clinvar/clinvar.vcf.gz", comment_prefix="##", separator="\t", ignore_errors=True)
    .with_columns(
        chrom = 'chr' + pl.col("#CHROM").fill_null("NA").cast(str),
    
        # Look for "CLNSIG=", capture everything until the next semicolon
        clinical_significance = pl.col("INFO").str.extract(r"CLNSIG=([^;]+)", 1),
    )
    .drop(['#CHROM', 'ID', 'QUAL', 'FILTER'])
    .rename({
        "POS": "pos",
        "REF": "ref",
        "ALT": "alt",
        "INFO": "info",
    })
    .with_columns(
        id = pl.col("chrom") + ":" + pl.col("pos").cast(str) + ":" + pl.col("ref") + ":" + pl.col("alt"),

        # Look for "MC=", skip characters until pipe "|", capture text until next comma or semicolon
        variant_region = pl.col("info").str.extract(r"MC=[^|]+\|([^;,]+)", 1),
    )
    .select(['id', 'clinical_significance'])
    .unique()
)

clinvar

In [ ]:
clinvar['clinical_significance'].value_counts(sort=True).head(10)

In [ ]:
# anno_with_popeve = pl.scan_parquet(f"{LOCAL_DATA_DIR}/annotations_with_all_with_clinpred_bayesdel_popeve.parquet")
anno_with_popeve = pl.scan_parquet(f"{LOCAL_DATA_DIR}/annotations_with_all.parquet").drop('spliceai_pred')
anno_with_popeve.head().collect()

In [ ]:
(
    anno_with_popeve
    .join(
        clinvar.lazy(),
        on='id',
        how='left'
    )
    .sink_parquet(f"{LOCAL_DATA_DIR}/annotations_with_all_with_clinvar.parquet", engine='streaming')
)

---
# Save and upload

In [ ]:
anno_with_all = pl.scan_parquet(f"{LOCAL_DATA_DIR}/annotations_with_all_with_clinvar.parquet")

encode_cols = [c for c in anno_with_all.collect_schema().names() if 'encode']
# Check is encode columns are present in the new dataframe
if len(encode_cols) == 0:
    print("Adding ENCODE annotations...")
    RAP_ANNO_DIR = "project-REDACTED:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated"
    ANNO_FILE = "annotations_with_all_old.parquet"
    !dx download {RAP_ANNO_DIR}/{ANNO_FILE} -o {LOCAL_DATA_DIR}/{ANNO_FILE}

    anno_enc = (
        pl.scan_parquet(f"{LOCAL_DATA_DIR}/{ANNO_FILE}")
        .select(['id', 'region', 'not_annotated_in_encode', 'encode_dels', 'encode_ca-ctcf', 'encode_ca', 'encode_ca-h3k4me3', 'encode_tf', 'encode_ca-tf', 'encode_pels', 'encode_pls'])
        .collect()
    )

    anno_with_all = (
        anno_with_all
        .join(
            anno_enc.lazy(),
            on=['id', 'region'],
            how='left'
        )
    )

print("Dropping duplicates and writing final dataframe...")
(
    anno_with_all
    .unique()
    .sink_parquet(f"{LOCAL_DATA_DIR}/annotations_with_all.parquet")
)

In [ ]:
!dx upload {LOCAL_DATA_DIR}/annotations_with_all.parquet --path {RAP_DATA_DIR}/